# Agentic AI and RAGs ?

Using only open source frameworks.
- Tiny KB with FAISS + FakeEmbeddings
- Free Wikipedia utility (no API keys)
- Rule-based planner
- Stub summarizer with optional tiny HF model (sshleifer/tiny-gpt2)
Run all cells top to bottom in Colab.

In [1]:
!pip install -q langchain langchain-community faiss-cpu wikipedia transformers accelerate sentencepiece

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 1) Build the KB retriever

In [2]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FakeEmbeddings

kb_docs = [
    Document(
        page_content="""
Agentic systems reason step-by-step about which tools to call instead of invoking tools blindly.
Their core loop is: (1) interpret the user goal, (2) inspect available context, (3) decide whether tools are needed,
(4) call one or more tools in a planned sequence, and (5) synthesize an answer grounded in the tool results.

Key principles for agentic behavior:
- Prefer direct answering from retrieved or provided context when sufficient; avoid unnecessary tool calls.
- Choose the simplest tool or minimal set of tools that can satisfy the user’s goal.
- When multiple tools are needed, sketch a short plan (ordered tool calls with dependencies) before executing.
- After each tool call, update an internal scratchpad and re-evaluate whether more tools are still necessary.
- Handle tool failures gracefully: inspect error messages, adjust inputs, or fall back to alternative tools.

Agentic systems must also enforce safety and constraints:
- Do not call tools that violate policy or appear unsafe given the user’s intent.
- Throttle repetitive or redundant calls that would waste resources without improving the answer.
- Be explicit in reasoning about trade-offs (e.g., precision vs. latency) when picking tools.
""".strip(),
        metadata={"source": "kb:agentic_concept"},
    ),
    Document(
        page_content="""
Retrievers fetch grounding passages from a knowledge base and are the primary interface to internal documents.
Given a user query, the system should: (1) normalize and possibly expand the query, (2) retrieve top-k candidates,
(3) read them carefully, and (4) base the answer primarily on those passages.

Best practices for using retrievers:
- Start with a moderately small k (e.g., 3–10) to keep context focused; increase k only if evidence is clearly missing.
- Reformulate the query if results look noisy or off-topic (e.g., add constraints, synonyms, or key entities).
- Use metadata filters (dates, document types, domains) when available to narrow down to the most relevant sources.
- When multiple passages agree, highlight the shared core facts; when they conflict, surface the disagreement explicitly.

Citing retriever outputs:
- Quote or paraphrase the most important sentences and explicitly cite their source identifiers (e.g., document IDs).
- Avoid cherry-picking single passages that contradict the majority of evidence without acknowledging the discrepancy.
- If retrieved passages do not directly support a precise answer, say so instead of guessing, and suggest follow-up queries.
""".strip(),
        metadata={"source": "kb:retrievers"},
    ),
    Document(
        page_content="""
Wikipedia is a broad-coverage, free fallback when the curated knowledge base lacks coverage or appears incomplete.
It should not be the first resort when high-quality internal documents exist, but can complement them for general facts,
background context, or definitions.

Guidelines for using Wikipedia:
- First attempt retrieval over the internal KB; only query Wikipedia if internal retrieval yields little or no relevant evidence.
- Use Wikipedia mainly for widely accepted, non-controversial information (basic facts, dates, definitions, high-level overviews).
- For critical or sensitive topics (e.g., medical, legal, or contentious political claims), treat Wikipedia as a starting point,
  not as the final authority; cross-check with more authoritative sources when possible.
- Pay attention to section structure: lead sections usually contain stable, high-level facts; deeper sections may be more speculative.

Communicating Wikipedia usage to users:
- Clearly indicate when information is derived from Wikipedia or similar external encyclopedic sources.
- Prefer concise, well-supported statements over niche details that rely on poorly sourced or disputed claims.
- If a Wikipedia article appears inconsistent or low quality, either omit those details or explicitly flag the uncertainty.
""".strip(),
        metadata={"source": "kb:wikipedia_tip"},
    ),
    Document(
        page_content="""
When evidence is thin, ambiguous, or conflicting, the system must be transparent about uncertainty instead of fabricating detail.
Honesty requires separating what is directly supported by sources from what is inferred, and clearly expressing confidence levels.

Practical rules for honest behavior:
- Always distinguish between: (a) facts directly supported by retrieved passages, (b) reasonable inferences, and (c) speculation.
- If key information is missing, explicitly say that the available sources do not answer the question fully.
- Do not invent citations, URLs, names, or numbers to fill gaps; if you cannot support them, do not present them as factual.
- When sources conflict, summarize each position, highlight the disagreement, and avoid taking a definitive stance without justification.

Helpful ways to express uncertainty:
- Use phrases like “the retrieved sources indicate…”, “based on the available evidence…”, or “the documents do not specify…”.
- Suggest concrete next steps: clarifying the question, expanding or narrowing retrieval, or consulting domain experts or
  authoritative references.
- Prefer being conservatively accurate over confidently wrong, even if that means providing a partial answer.
""".strip(),
        metadata={"source": "kb:honesty"},
    ),
    Document(
        page_content="""
Answers should be concise, focused, and well-structured, typically within 2–4 sentences for straightforward questions.
Lead with the main conclusion, then briefly justify it using the most relevant evidence and clearly marked citations.

Style and structure guidelines:
- Directly answer the user’s question in the first sentence whenever possible.
- Avoid long preambles, restating the entire question, or narrating internal reasoning unless explicitly asked.
- Use clear, concrete language; minimize jargon and define necessary technical terms in simple words.
- For complex topics, provide a short high-level summary first, then optionally a brief, structured breakdown (e.g., bullet points).

Citation and formatting:
- Cite the minimal set of sources that substantively support the key claims, rather than listing all retrieved documents.
- Integrate citations inline, near the claims they support, to make it easy to trace evidence.
- Keep formatting clean: short paragraphs, occasional bullets for lists, and no unnecessary verbosity.

When to expand beyond 2–4 sentences:
- If the user explicitly asks for a detailed explanation, tutorial, or step-by-step reasoning.
- If the topic is safety-critical and requires more context, caveats, or assumptions to avoid misinterpretation.
In all cases, maintain clarity and relevance as the primary goals.
""".strip(),
        metadata={"source": "kb:style"},
    ),
]

embeddings = FakeEmbeddings(size=256)
vs = FAISS.from_documents(kb_docs, embeddings)
retriever = vs.as_retriever(search_kwargs={"k": 3})
print("KB ready with", len(kb_docs), "docs")

/tmp/ipykernel_2637/3710946191.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


KB ready with 5 docs


## 2) Open source external tool: Wikipedia search

In [3]:
from langchain_community.utilities import WikipediaAPIWrapper

wiki = WikipediaAPIWrapper(lang="en", top_k_results=2, doc_content_chars_max=1000)

def wiki_search(query: str, k: int = 2):
    """Return Wikipedia article titles and short summaries; report network errors."""
    try:
        pages = wiki.load(query)[:k]
        snippets = [
            {"title": page.metadata.get("title", "Unknown"),
             "summary": page.page_content[:700].strip()}
            for page in pages if page.page_content.strip()
        ]
        return snippets, None
    except Exception as exc:
        return [], f"Wikipedia lookup failed: {type(exc).__name__}: {exc}"

print("Wikipedia helper ready; live requests run during the quick check.")


Wikipedia helper ready; live requests run during the quick check.


## 3) Simple planner (rule-based)

In [4]:
import re

kb_keywords = ["agentic", "retriever", "retrieval", "citation", "cite", "ground", "transparen", "honest", "evidence", "rag"]

def plan(question: str):
    """Route known KB topics to the local retriever; other questions to Wikipedia."""
    text = question.casefold()
    matched = [word for word in kb_keywords if re.search(r"\b" + re.escape(word), text)]
    return {"action": "kb" if matched else "wiki", "matched_keywords": matched}

print(plan("How to ground answers?"))
print(plan("Who created Python?"))


{'action': 'kb', 'matched_keywords': ['ground']}
{'action': 'wiki', 'matched_keywords': []}


## 4) Answer function with stub or tiny HF model

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_models.fake import FakeListChatModel

prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer using only the evidence below. "
    "If the evidence is insufficient, say so and suggest a follow-up query. "
    "Cite factual claims using the exact supplied source IDs.\n"
    "Question: {question}\nContext: {context}\nWikipedia: {wiki}\nAnswer:"
)

# Optional local generation. Not required for the default, lightweight Colab run.
def get_tiny_generator(model_id: str = "sshleifer/tiny-gpt2"):
    from transformers import pipeline
    return pipeline("text-generation", model=model_id, device=-1)

def summarize_with_tiny(prompt_text: str, max_new_tokens: int = 120):
    generator = get_tiny_generator()
    result = generator(prompt_text, max_new_tokens=max_new_tokens, do_sample=False,
                       return_full_text=False, pad_token_id=50256)
    return result[0]["generated_text"].strip()

def answer_question(question: str, use_tiny_model: bool = False):
    pl = plan(question)
    # FakeEmbeddings are random and do not measure semantic relevance. Use the
    # required top-3 retriever, then avoid claiming it proves unsupported facts.
    docs = retriever.invoke(question) if pl["action"] == "kb" else []
    wiki_snips, wiki_err = (wiki_search(question) if pl["action"] == "wiki" else ([], None))
    context_text = "\n".join(
        f"[{d.metadata.get('source', 'kb:unknown')}] {d.page_content}" for d in docs
    ) or "No KB context."
    wiki_text = "\n".join(
        f"[wiki:{s['title'].replace(' ', '_')}] {s['summary']}" for s in wiki_snips
    ) or "No Wikipedia snippets."
    messages = prompt.format_messages(question=question, context=context_text, wiki=wiki_text)
    kb_sources = [d.metadata.get("source", "kb:unknown") for d in docs]
    wiki_sources = [s["title"] for s in wiki_snips]

    if not docs and not wiki_snips:
        final_answer = ("Insufficient evidence to answer reliably. Try a more specific "
                        "follow-up question or retry the Wikipedia search.")
    elif use_tiny_model:
        # Tiny GPT-2 is illustrative, not reliable for factual attribution.
        generated = summarize_with_tiny(messages[0].content)
        final_answer = generated + "\nSources consulted: " + ", ".join(
            [f"[{source}]" for source in kb_sources] +
            [f"[wiki:{title.replace(' ', '_')}]" for title in wiki_sources]
        )
    else:
        # Exercise requires a FakeListChatModel. Use evidence-derived responses
        # so that the stub does not invent unsupported facts or citations.
        if docs:
            # KB passages contain broad guidance, not always a direct answer.
            chosen = next((d for d in docs if any(
                keyword in d.page_content.casefold() for keyword in pl["matched_keywords"]
            )), None)
            if chosen is None:
                response = ("The retrieved passages do not directly support a specific answer. "
                            "Try a narrower question about the KB topics.")
            else:
                excerpt = " ".join(chosen.page_content.split())[:260]
                response = (f"The KB states: {excerpt}... "
                            f"[{chosen.metadata['source']}] "
                            "For more detail, ask a narrower follow-up question.")
        else:
            first = wiki_snips[0]
            excerpt = " ".join(first["summary"].split())[:280]
            response = (f"Wikipedia reports: {excerpt}... "
                        f"[wiki:{first['title'].replace(' ', '_')}] "
                        "Check the full article for additional context.")
        stub = FakeListChatModel(responses=[response])
        final_answer = stub.invoke(messages).content

    return {"plan": pl, "kb_sources": kb_sources, "wiki_sources": wiki_sources,
            "wiki_error": wiki_err, "answer": final_answer}


## 5) Quick check on sample questions

In [6]:
tests = [
    "How can agentic systems ground answers with citations?",  # KB
    "Who created Python programming language?",                # Wikipedia
    "What happened in the mysterious event at XYZ-999?",       # ambiguous / thin evidence
]

for q in tests:
    try:
        res = answer_question(q, use_tiny_model=False)
        print("\nQ:", q)
        print("Plan:", res["plan"])
        print("KB sources:", res["kb_sources"])
        print("Wiki sources:", res["wiki_sources"])
        if res["wiki_error"]:
            print("Wiki warning:", res["wiki_error"])
        print("Answer:", res["answer"])
        assert res["plan"]["action"] in {"kb", "wiki"}
        assert isinstance(res["answer"], str) and res["answer"]
    except Exception as exc:
        print(f"Question failed gracefully: {type(exc).__name__}: {exc}")



Q: How can agentic systems ground answers with citations?
Plan: {'action': 'kb', 'matched_keywords': ['agentic', 'citation', 'ground']}
KB sources: ['kb:honesty', 'kb:wikipedia_tip', 'kb:style']
Wiki sources: []
Answer: The KB states: When evidence is thin, ambiguous, or conflicting, the system must be transparent about uncertainty instead of fabricating detail. Honesty requires separating what is directly supported by sources from what is inferred, and clearly expressing confidence levels.... [kb:honesty] For more detail, ask a narrower follow-up question.

Q: Who created Python programming language?
Plan: {'action': 'wiki', 'matched_keywords': []}
KB sources: []
Wiki sources: ['Python (programming language)', 'General-purpose programming language']
Answer: Wikipedia reports: Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation, an extensive ("batteries-included") standar